# Korea PSR Valuation - 디버깅 & 분석 노트북

## 목적
단일 종목 코드를 입력하여 전체 예측 과정을 단계별로 실행하고, 각 단계의 데이터프레임을 확인할 수 있습니다.

## 주요 기능
1. 매출 데이터 추출 및 전처리 과정 확인
2. SARIMA, ETS, Theta, LSTM, Prophet 모델별 예측 결과 비교
3. PSR 시계열 생성 및 예측 확인
4. 최종 Valuation 계산 결과 검토
5. DB 저장 없이 순수 분석만 수행

## 사용법
1. 모든 설정 및 함수 정의 셀 실행
2. "종목 코드 입력" 섹션에서 분석할 종목 입력
3. "단계별 실행" 섹션의 셀들을 순서대로 실행
4. 각 단계마다 생성되는 데이터프레임 확인


## 1. 라이브러리 Import

In [1]:
import gc
import traceback
import pandas as pd
import numpy as np
from datetime import datetime
import sys, os
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

# 출력 설정
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}' if abs(x) >= 1 else f'{x:.6f}')

## 2. 인코딩 설정

In [2]:
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass

os.environ.setdefault("PYTHONIOENCODING", "utf-8")

'UTF-8'

## 3. 예측 날짜 설정

In [3]:
# 예측 날짜 설정
FORECAST_CONFIG = {
    'revenue_start_date': '2025-12-31',  # Revenue 예측 시작일 (분기말)
    'psr_start_date': '2026-01-31',      # PSR 예측 시작일 (월말)
    'horizon_quarters': 6,                # Revenue 예측 분기 수
    'psr_horizon_months': 24,            # PSR 예측 월 수
}

# 자동 모드로 전환하려면 아래 주석 해제
# FORECAST_CONFIG = {
#     'revenue_start_date': None,
#     'psr_start_date': None,
#     'horizon_quarters': 6,
#     'psr_horizon_months': 24,
# }

print("📅 예측 설정:")
print(f"  Revenue 예측 시작일: {FORECAST_CONFIG['revenue_start_date']}")
print(f"  PSR 예측 시작일: {FORECAST_CONFIG['psr_start_date']}")
print(f"  Revenue 예측 분기: {FORECAST_CONFIG['horizon_quarters']}분기")
print(f"  PSR 예측 월수: {FORECAST_CONFIG['psr_horizon_months']}개월")

📅 예측 설정:
  Revenue 예측 시작일: 2025-12-31
  PSR 예측 시작일: 2026-01-31
  Revenue 예측 분기: 6분기
  PSR 예측 월수: 24개월


## 4. 사용자 정의 모듈 Import

In [4]:
from DATA.stock_invest_function import *
from get_market_cap_by_ticker import get_market_cap_by_ticker

# 노트북과 동일한 핵심 함수들 직접 import
from DATA.universal_ts_forecast_function import (
    forecast_ets,
    forecast_theta,
    infer_freq_alias,
    seasonal_periods_from_freq,
    ensure_datetime_index_df
)

# 기존 모듈들
from get_fs_data_by_ticker_v2 import fetch_table_data
from get_hscode_processed_data import get_hscode_processed_data
from get_monthly_export_data import extract_monthly_exog_var
from get_revenue_export_joined_table import get_revenue_export_joined_table
from sarima_endog_forecast import forecast_endog_with_optional_exog
from sarima_endog_forecast import forecast_endog_fill_tail
from get_forecasted_revenue_df import build_forecast_df_from_out
from revenue_forecast_all_package import forecast_revenue_lstm, forecast_revenue_prophet
from get_revenue_ttm_df import get_revenue_ttm_df
from get_psr_from_mc_and_rev import build_psr_series
from psr_forecast_runner import forecast_psr_all_models
from valuation_forecast import compute_valuation_forecast

print("✓ 모듈 import 완료")


✓ 모듈 import 완료


## 5. 데이터베이스 설정

In [5]:
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

EXOG_Q_TRANSFORM = "pct"

print("✓ DB 설정 완료")

✓ DB 설정 완료


## 6. 핵심 함수 정의

### 6.1 매출 데이터 추출 함수

In [6]:
def extract_quarterly_revenue_notebook_style(
        db_info: dict,
        ticker: str,
        fs_df: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    """노트북 스타일의 매출 데이터 추출"""
    import pymysql
    
    ticker_clean = ticker.lstrip('A').zfill(6)
    ticker_dg = 'A' + ticker_clean
    
    # 1) DataGuide 데이터
    if fs_df is None:
        fs_df = fetch_table_data(db_info, "korea_fs_data")
    
    revenue_dg = fs_df[
        (fs_df['symbol'] == ticker_dg) &
        (fs_df['indicator'] == '매출액(천원)')
    ].copy()
    
    if not revenue_dg.empty:
        revenue_from_dg = revenue_dg[['date', 'value']].copy()
        revenue_from_dg['date'] = pd.to_datetime(revenue_from_dg['date'], errors='coerce')
        revenue_from_dg['value'] = pd.to_numeric(revenue_from_dg['value'], errors='coerce') * 1000
        revenue_from_dg = revenue_from_dg.dropna()
    else:
        revenue_from_dg = pd.DataFrame(columns=['date', 'value'])
    
    # 2) DART 데이터
    conn = pymysql.connect(
        host=db_info["host"],
        port=db_info["port"],
        user=db_info["user"],
        password=db_info["password"],
        db=db_info.get("db", db_info.get("database")),
        charset="utf8mb4",
    )
    
    try:
        sql = """
              SELECT *
              FROM korea_fs_data_from_DART
              WHERE ticker = %s
                AND account_id IN ('ifrs_Revenue', 'ifrs-full_Revenue')
              ORDER BY bsns_year, report_date
              """
        revenue_df = pd.read_sql(sql, conn, params=[ticker_clean])
        
        if not revenue_df.empty:
            revenue_df = _adjust_fy_to_q4(revenue_df)
            revenue_from_dart = revenue_df[['report_date', 'thstrm_amount']].copy()
            revenue_from_dart['report_date'] = pd.to_datetime(revenue_from_dart['report_date'], errors='coerce')
            revenue_from_dart['thstrm_amount'] = pd.to_numeric(revenue_from_dart['thstrm_amount'], errors='coerce')
            revenue_from_dart = revenue_from_dart.dropna()
        else:
            revenue_from_dart = pd.DataFrame(columns=['report_date', 'thstrm_amount'])
    finally:
        conn.close()
    
    # 3) 결합 (keep='first' - DataGuide 우선)
    revenue_from_dg.columns = ['date', 'revenue']
    revenue_from_dart.columns = ['date', 'revenue']
    
    revenue_combined = pd.concat(
        [revenue_from_dg, revenue_from_dart],
        axis=0
    ).drop_duplicates(
        subset=['date'],
        keep='first'
    ).sort_values('date').reset_index(drop=True)
    
    if revenue_combined.empty:
        return pd.DataFrame(columns=['revenue', 'year', 'quarter', 'year_quarter', 'symbol'])
    
    # 4) 날짜/분기 파생 변수 생성
    revenue_combined['year'] = revenue_combined['date'].dt.year
    revenue_combined['quarter'] = revenue_combined['date'].dt.quarter
    revenue_combined['year_quarter'] = (
            revenue_combined['year'].astype(str) + "Q" +
            revenue_combined['quarter'].astype(str)
    )
    revenue_combined['symbol'] = ticker_dg
    
    # 5) Date 인덱스 설정
    revenue_combined = revenue_combined.set_index('date').sort_index()
    revenue_combined.index.name = 'date'
    
    return revenue_combined[['revenue', 'year', 'quarter', 'year_quarter', 'symbol']].copy()


def _adjust_fy_to_q4(df: pd.DataFrame) -> pd.DataFrame:
    """FY를 Q4로 변환"""
    result_df = df.copy()
    for year in result_df['bsns_year'].unique():
        year_mask = result_df['bsns_year'] == year
        fy_mask = year_mask & (result_df['quarter'] == 'FY')
        
        if fy_mask.any():
            fy_amount = result_df.loc[fy_mask, 'thstrm_amount'].iloc[0]
            q123_mask = year_mask & result_df['quarter'].isin(['Q1', 'Q2', 'Q3'])
            q123_sum = result_df.loc[q123_mask, 'thstrm_amount'].sum()
            pure_q4 = fy_amount - q123_sum
            
            result_df.loc[fy_mask, 'quarter'] = 'Q4'
            result_df.loc[fy_mask, 'thstrm_amount'] = pure_q4
    
    return result_df

print("✓ 매출 데이터 추출 함수 정의 완료")

✓ 매출 데이터 추출 함수 정의 완료


### 6.2 PeriodIndex 변환 함수

In [7]:
def convert_to_period_index(df: pd.DataFrame, value_col: str = 'revenue') -> pd.Series:
    """DatetimeIndex → PeriodIndex 변환"""
    df = df.copy()
    
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, errors='coerce')
    
    df = df[~df.index.isna()].sort_index()
    
    df['year'] = df.index.year
    df['quarter'] = df.index.quarter
    df['period'] = df['year'].astype(str) + 'Q' + df['quarter'].astype(str)
    
    df['period'] = pd.PeriodIndex(df['period'], freq='Q')
    df = df.set_index('period').sort_index()
    
    if df.index.duplicated().any():
        print("  ⚠️  중복 인덱스 발견, 마지막 값 유지")
        df = df[~df.index.duplicated(keep='last')]
    
    if value_col in df.columns:
        series = df[value_col].copy()
    else:
        raise ValueError(f"'{value_col}' 컬럼이 DataFrame에 없습니다.")
    
    return series

print("✓ PeriodIndex 변환 함수 정의 완료")

✓ PeriodIndex 변환 함수 정의 완료


### 6.3 외생변수 변환 함수

In [8]:
def _transform_exog_col(df: pd.DataFrame, col: str, periods: int = 4, method: str = "pct") -> pd.DataFrame:
    """외생변수 변환"""
    df = df.copy()
    
    if method == "pct":
        df[col] = df[col].pct_change(periods=periods)
    elif method == "diff":
        df[col] = df[col].diff(periods=periods)
    
    df = df.dropna(subset=[col])
    return df

print("✓ 외생변수 변환 함수 정의 완료")

✓ 외생변수 변환 함수 정의 완료


### 6.4 메모리 정리 함수

In [9]:
def _cleanup(*args):
    """메모리 정리"""
    for obj in args:
        if obj is not None:
            del obj
    gc.collect()

print("✓ 메모리 정리 함수 정의 완료")

✓ 메모리 정리 함수 정의 완료


### 6.5 보조 함수들

In [10]:
def forecast_ets_notebook_style(series: pd.Series, forecast_horizon: int = 9,
                                forecast_start_date: str = None) -> pd.DataFrame:
    """노트북 스타일 ETS 예측"""
    from date_standardization import get_standard_quarter_dates
    
    # 계절성 파라미터 계산
    if isinstance(series.index, pd.PeriodIndex):
        freq = series.index.freq
        m = seasonal_periods_from_freq(freq)
        inferred_freq = freq
    else:
        inferred_freq = infer_freq_alias(series.index)
        m = seasonal_periods_from_freq(inferred_freq)
    
    print(f"  - ETS 예측 중... (freq={inferred_freq}, m={m})")
    
    # forecast_ets 직접 호출
    ets_result = forecast_ets(
        y=series,
        forecast_horizon=forecast_horizon,
        m=m,
        try_transforms=True
    )
    
    forecast_values = ets_result['forecast']
    
    # 수동 시작일이 지정되었으면 사용
    if forecast_start_date is not None:
        last_actual = pd.Timestamp(series.index.max().to_timestamp('Q'))
        datetime_index = get_standard_quarter_dates(
            last_actual.strftime('%Y-%m-%d'),
            forecast_horizon,
            forecast_start_date=forecast_start_date
        )
    else:
        forecast_index = ets_result.get('forecast_index', None)
        
        if forecast_index is None:
            last_period = series.index.max()
            forecast_index = pd.period_range(
                last_period + 1,
                periods=forecast_horizon,
                freq='Q'
            )
        
        if isinstance(forecast_index, pd.PeriodIndex):
            datetime_index = forecast_index.to_timestamp('Q')
        else:
            datetime_index = pd.DatetimeIndex(forecast_index)
    
    return pd.DataFrame(
        forecast_values,
        index=datetime_index,
        columns=['revenue']
    )


def forecast_theta_notebook_style(series: pd.Series, forecast_horizon: int = 9,
                                  forecast_start_date: str = None) -> pd.DataFrame:
    """노트북 스타일 Theta 예측"""
    from date_standardization import get_standard_quarter_dates
    
    # 계절성 파라미터 계산
    if isinstance(series.index, pd.PeriodIndex):
        freq = series.index.freq
        m = seasonal_periods_from_freq(freq)
        inferred_freq = freq
    else:
        inferred_freq = infer_freq_alias(series.index)
        m = seasonal_periods_from_freq(inferred_freq)
    
    print(f"  - Theta 예측 중... (freq={inferred_freq}, m={m})")
    
    theta_result = forecast_theta(
        y=series,
        forecast_horizon=forecast_horizon,
        m=m,
        try_transforms=True
    )
    
    forecast_values = theta_result['forecast']
    
    if forecast_start_date is not None:
        last_actual = pd.Timestamp(series.index.max().to_timestamp('Q'))
        datetime_index = get_standard_quarter_dates(
            last_actual.strftime('%Y-%m-%d'),
            forecast_horizon,
            forecast_start_date=forecast_start_date
        )
    else:
        forecast_index = theta_result.get('forecast_index', None)
        
        if forecast_index is None:
            last_period = series.index.max()
            forecast_index = pd.period_range(
                last_period + 1,
                periods=forecast_horizon,
                freq='Q'
            )
        
        if isinstance(forecast_index, pd.PeriodIndex):
            datetime_index = forecast_index.to_timestamp('Q')
        else:
            datetime_index = pd.DatetimeIndex(forecast_index)
    
    return pd.DataFrame(
        forecast_values,
        index=datetime_index,
        columns=['revenue']
    )

print("✓ 보조 함수 정의 완료")

✓ 보조 함수 정의 완료


### 6.6 매출 예측 함수 (노트북 스타일)

In [11]:
def forecast_revenue_models_notebook_style(
        final_combined_data: pd.DataFrame,
        exog_df: Optional[pd.DataFrame],
        horizon: int = 6,
        hs_code: Optional[str] = None
):
    """노트북 스타일의 매출 예측"""
    
    # TTM 계산을 위해 추가 분기 예측
    extended_horizon = horizon + 3
    
    # 1) SARIMA (exog 없음)
    try:
        out_noexog = forecast_endog_with_optional_exog(
            combined_df=final_combined_data,
            horizon=extended_horizon,
            hs_code=None
        )
        sarima_noexog = build_forecast_df_from_out(out_noexog, combined_df=final_combined_data)
        
        # 안전성 검증
        if sarima_noexog is not None and not sarima_noexog.empty:
            sarima_values = sarima_noexog.iloc[:, 0]
            if (sarima_values < 0).any() or sarima_values.isna().any() or (sarima_values > 1e15).any():
                print(f"  ⚠️  SARIMA(no exog) 예측 불안정 - 제외")
                sarima_noexog = None
    except Exception as e:
        print(f"  ⚠️  SARIMA(no exog) 실패: {e}")
        sarima_noexog = None
    
    # 2) SARIMA (exog 포함)
    sarima_exog = None
    if hs_code:
        try:
            out_exog = forecast_endog_fill_tail(final_combined_data, hs_code=hs_code)
            sarima_exog = build_forecast_df_from_out(out_exog, combined_df=final_combined_data)
            
            # 안전성 검증
            if sarima_exog is not None and not sarima_exog.empty:
                sarima_exog_values = sarima_exog.iloc[:, 0]
                if (sarima_exog_values < 0).any() or sarima_exog_values.isna().any() or (sarima_exog_values > 1e15).any():
                    print(f"  ⚠️  SARIMA(exog) 예측 불안정 - 제외")
                    sarima_exog = None
        except Exception as e:
            print(f"  ⚠️  SARIMA(exog) 실패: {e}")
    
    # 3) ETS - 노트북 방식
    try:
        series = convert_to_period_index(final_combined_data, value_col='endog_var')
        print(f"  시계열 마지막 기간: {series.index.max()}")
        
        ets_df = forecast_ets_notebook_style(
            series,
            forecast_horizon=extended_horizon,
            forecast_start_date=FORECAST_CONFIG.get('revenue_start_date')
        )
    except Exception as e:
        print(f"  ⚠️  ETS 실패: {e}")
        ets_df = None
    
    # 4) Theta - 노트북 방식
    try:
        theta_df = forecast_theta_notebook_style(
            series,
            forecast_horizon=extended_horizon,
            forecast_start_date=FORECAST_CONFIG.get('revenue_start_date')
        )
    except Exception as e:
        print(f"  ⚠️  Theta 실패: {e}")
        theta_df = None
    
    # 5) LSTM
    try:
        lstm_df = forecast_revenue_lstm(final_combined_data, horizon=extended_horizon, lookback=12)
    except Exception as e:
        print(f"  ⚠️  LSTM 실패: {e}")
        lstm_df = None
    
    # 6) Prophet
    try:
        prophet_df = forecast_revenue_prophet(final_combined_data, horizon=extended_horizon)
    except Exception as e:
        print(f"  ⚠️  Prophet 실패: {e}")
        prophet_df = None
    
    return (sarima_noexog, sarima_exog, ets_df, theta_df, lstm_df, prophet_df)

print("✓ 매출 예측 함수 정의 완료")

✓ 매출 예측 함수 정의 완료


### 6.7 최종 매출 DataFrame 생성 함수

In [12]:
# get_revenue_ttm_df 함수를 그대로 사용
# 이 함수는 이미 import되어 있음

print("✓ 최종 매출 DataFrame 생성 함수: get_revenue_ttm_df 사용")

✓ 최종 매출 DataFrame 생성 함수: get_revenue_ttm_df 사용


---
## 7. 종목 코드 입력 및 초기 설정

여기서 분석할 종목 코드를 입력하세요.


In [21]:
# 분석할 종목 코드 입력 (A 접두사 제외)
TICKER = '000660'  # 예: SK하이닉스

# HS Code (수출 데이터 사용 시, 없으면 None)
HS_CODE = None  # 예: '8542' 또는 None

# 기타 설정
VALUE_START_DATE = None  # Valuation 시작 날짜 (None이면 자동)
FORECAST_DATE = None     # 예측 기준일 (None이면 오늘)

print(f"🎯 분석 대상 종목: {TICKER}")
print(f"   HS Code: {HS_CODE}")
print(f"   Valuation 시작일: {VALUE_START_DATE}")
print(f"   예측 기준일: {FORECAST_DATE}")

🎯 분석 대상 종목: 000660
   HS Code: None
   Valuation 시작일: None
   예측 기준일: None


---
## 8. 단계별 실행 및 데이터 확인

아래 셀들을 순서대로 실행하면서 각 단계의 결과를 확인하세요.


### Step 1: 매출 데이터 추출

In [22]:
print("=" * 70)
print("Step 1: 매출 데이터 추출")
print("=" * 70)

# Ticker 정규화
ticker_clean = TICKER.lstrip('A').zfill(6)
ticker_with_a = 'A' + ticker_clean

# DataGuide 데이터 로드
print("\n📊 DataGuide 데이터 로딩 중...")
fs_df = fetch_table_data(db_info, "korea_fs_data")
print(f"✓ {len(fs_df):,}행 로드 완료")

# 매출 데이터 추출
print(f"\n📈 {ticker_with_a} 매출 데이터 추출 중...")
df_rev = extract_quarterly_revenue_notebook_style(
    db_info=db_info,
    ticker=ticker_clean,
    fs_df=fs_df
)

if df_rev.empty:
    print(f"⚠️  {ticker_with_a} 매출 데이터가 없습니다!")
else:
    print(f"✓ 매출 데이터 추출 완료: {len(df_rev)}개 분기")
    print(f"  데이터 기간: {df_rev.index.min().date()} ~ {df_rev.index.max().date()}")
    print(f"\n최근 5개 분기 매출:")
    display(df_rev.tail())

Step 1: 매출 데이터 추출

📊 DataGuide 데이터 로딩 중...
✓ 6,683,588행 로드 완료

📈 A000660 매출 데이터 추출 중...
✓ 매출 데이터 추출 완료: 87개 분기
  데이터 기간: 2004-03-31 ~ 2025-09-30

최근 5개 분기 매출:


,revenue,year,quarter,year_quarter,symbol
date,,,,,
2024-09-30,"17,573,069,000,000.00",2024,3,2024Q3,A000660
2024-12-31,"19,767,035,000,000.00",2024,4,2024Q4,A000660
2025-03-31,"17,639,141,000,000.00",2025,1,2025Q1,A000660
2025-06-30,"22,231,952,000,000.00",2025,2,2025Q2,A000660
2025-09-30,"24,448,929,000,000.00",2025,3,2025Q3,A000660


### Step 2: 데이터 전처리 및 결합

In [23]:
print("=" * 70)
print("Step 2: 데이터 전처리 및 결합")
print("=" * 70)

# 외생변수가 있으면 exog_df 사용, 없으면 None
exog_df_for_join = exog_df if 'exog_df' in dir() and exog_df is not None else None

# 결합 테이블 생성
print("\n📊 매출-수출 데이터 결합 중...")
combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
    df_rev=df_rev, 
    df_exog=exog_df_for_join, 
    join_how="outer", 
    fill_exog="ffill"
)

# 2015년 이후 데이터만 사용
final_combined_data = final_combined_data.loc['2015-01-01':]

print(f"✓ 결합 완료")
print(f"  전체 행 수: {len(final_combined_data)}")
print(f"  기간: {final_combined_data.index.min().date()} ~ {final_combined_data.index.max().date()}")
print(f"  컬럼: {list(final_combined_data.columns)}")

print(f"\n최근 데이터:")
display(final_combined_data.tail())

Step 2: 데이터 전처리 및 결합

📊 매출-수출 데이터 결합 중...
✓ 결합 완료
  전체 행 수: 43
  기간: 2015-03-31 ~ 2025-09-30
  컬럼: ['endog_var', 'exog_var']

최근 데이터:


,endog_var,exog_var
Date,,
2024-09-30,"17,573,069,000,000.00",NaN
2024-12-31,"19,767,035,000,000.00",NaN
2025-03-31,"17,639,141,000,000.00",NaN
2025-06-30,"22,231,952,000,000.00",NaN
2025-09-30,"24,448,929,000,000.00",NaN


In [24]:
final_combined_data

,endog_var,exog_var
Date,,
2015-03-31,"4,818,341,000,000.00",NaN
2015-06-30,"4,638,579,000,000.00",NaN
2015-09-30,"4,925,034,000,000.00",NaN
2015-12-31,"4,416,044,000,000.00",NaN
2016-03-31,"3,655,717,000,000.00",NaN
2016-06-30,"3,940,927,000,000.00",NaN
2016-09-30,"4,243,645,000,000.00",NaN
2016-12-31,"5,357,686,000,000.00",NaN
2017-03-31,"6,289,518,000,000.00",NaN


### Step 3: 수출 데이터 처리 (외생변수)

In [25]:
print("=" * 70)
print("Step 3: 수출 데이터 처리")
print("=" * 70)

exog_df = None

if HS_CODE:
    print(f"\n📦 HS Code {HS_CODE} 수출 데이터 수집 중...")
    try:
        df_export = get_hscode_processed_data(db_info, hs_code=HS_CODE)
        
        if df_export is not None and isinstance(df_export, dict):
            print(f"✓ 수출 데이터 수집 완료")
            
            # 분기별 데이터 추출
            if 'quarterly' in df_export and df_export['quarterly'] is not None:
                exog_df = df_export['quarterly']
                
                if 'exog_var' in exog_df.columns:
                    # 변환 적용
                    exog_df = _transform_exog_col(
                        df=exog_df,
                        col='exog_var',
                        periods=4,
                        method=EXOG_Q_TRANSFORM
                    )
                    
                    print(f"✓ 외생변수 변환 완료 ({EXOG_Q_TRANSFORM})")
                    print(f"  외생변수 길이: {len(exog_df)}")
                    print(f"\n최근 외생변수:")
                    display(exog_df.tail())
                else:
                    print("⚠️  exog_var 컬럼 없음")
            else:
                print("⚠️  분기별 데이터 없음")
        else:
            print("⚠️  수출 데이터 없음")
    except Exception as e:
        print(f"⚠️  수출 데이터 처리 실패: {e}")
        import traceback
        traceback.print_exc()
else:
    print("ℹ️  HS Code가 지정되지 않아 외생변수를 사용하지 않습니다.")

Step 3: 수출 데이터 처리
ℹ️  HS Code가 지정되지 않아 외생변수를 사용하지 않습니다.


### Step 4: 매출 예측 (6개 모델)

In [26]:
print("=" * 70)
print("Step 4: 매출 예측")
print("=" * 70)

horizon_quarters = FORECAST_CONFIG['horizon_quarters']

print(f"\n예측 분기 수: {horizon_quarters}")
print(f"예측 시작일: {FORECAST_CONFIG.get('revenue_start_date')}")

# 예측 실행
print("\n🔮 모델별 예측 실행 중...")
(rev_sarima_noexog, 
 rev_sarima_exog, 
 rev_ets_df, 
 rev_theta_df, 
 rev_lstm_df, 
 rev_prophet_df) = forecast_revenue_models_notebook_style(
    final_combined_data=final_combined_data,
    exog_df=exog_df if 'exog_df' in dir() else None,
    horizon=horizon_quarters,
    hs_code=HS_CODE
)

print("\n✓ 예측 완료")
print("\n📊 모델별 예측 결과:")
print(f"  SARIMA (no exog): {'✓' if rev_sarima_noexog is not None else '✗'}")
print(f"  SARIMA (exog):    {'✓' if rev_sarima_exog is not None else '✗'}")
print(f"  ETS:              {'✓' if rev_ets_df is not None else '✗'}")
print(f"  Theta:            {'✓' if rev_theta_df is not None else '✗'}")
print(f"  LSTM:             {'✓' if rev_lstm_df is not None else '✗'}")
print(f"  Prophet:          {'✓' if rev_prophet_df is not None else '✗'}")

Step 4: 매출 예측

예측 분기 수: 6
예측 시작일: 2025-12-31

🔮 모델별 예측 실행 중...
[메모리] forecast_sarima 실행 전: 5394.71 MB
[메모리] find_best_sarima_params 실행 전: 5394.71 MB
[메모리] find_best_sarima_params 실행 후: 5395.71 MB (변화: +1.00 MB)
[메모리] forecast_sarima 실행 후: 5395.71 MB (변화: +1.00 MB)
  시계열 마지막 기간: 2025Q3
  - ETS 예측 중... (freq=<QuarterEnd: startingMonth=12>, m=4)
[메모리] forecast_ets 실행 전: 5395.71 MB
[메모리] forecast_ets 실행 후: 5395.98 MB (변화: +0.26 MB)
  - Theta 예측 중... (freq=<QuarterEnd: startingMonth=12>, m=4)
[메모리] forecast_theta 실행 전: 5395.98 MB
[메모리] forecast_theta 실행 후: 5395.98 MB (변화: +0.00 MB)
[메모리] forecast_lstm 실행 전: 5395.98 MB


00:12:15 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_lstm 실행 후: 5425.96 MB (변화: +29.98 MB)
[메모리] forecast_prophet 실행 전: 5425.96 MB


00:12:15 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 5426.34 MB (변화: +0.38 MB)

✓ 예측 완료

📊 모델별 예측 결과:
  SARIMA (no exog): ✓
  SARIMA (exog):    ✗
  ETS:              ✓
  Theta:            ✓
  LSTM:             ✓
  Prophet:          ✓


In [27]:
rev_sarima_noexog

,revenue_with_noexog
2025-12-31,"26,541,426,722,388.04"
2026-03-31,"25,612,971,716,571.28"
2026-06-30,"29,678,475,919,022.06"
2026-09-30,"31,354,894,245,703.95"
2026-12-31,"33,194,417,728,529.56"
2027-03-31,"32,154,053,499,116.35"
2027-06-30,"36,170,051,971,862.72"
2027-09-30,"37,824,570,251,392.07"
2027-12-31,"39,654,405,722,988.53"


### Step 5: 개별 모델 예측 결과 확인

In [18]:
print("=" * 70)
print("Step 5: 개별 모델 예측 결과 확인")
print("=" * 70)

if rev_sarima_noexog:
    print("\n[SARIMA - No Exog]")
    fc_noexog = build_forecast_df_from_out(rev_sarima_noexog, col_name='forecast')
    display(fc_noexog)

if rev_sarima_exog:
    print("\n[SARIMA - With Exog]")
    fc_exog = build_forecast_df_from_out(rev_sarima_exog, col_name='forecast')
    display(fc_exog)

if rev_ets_df is not None:
    print("\n[ETS]")
    display(rev_ets_df)

if rev_theta_df is not None:
    print("\n[Theta]")
    display(rev_theta_df)

if rev_lstm_df is not None:
    print("\n[LSTM]")
    display(rev_lstm_df)

if rev_prophet_df is not None:
    print("\n[Prophet]")
    display(rev_prophet_df)

Step 5: 개별 모델 예측 결과 확인


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

### Step 6: 최종 매출 DataFrame 생성 (TTM 포함)

In [ ]:
print("=" * 70)
print("Step 6: 최종 매출 DataFrame 생성")
print("=" * 70)

# get_revenue_ttm_df 함수 사용 (원본 코드와 동일)
rev_final = get_revenue_ttm_df(
    df_rev=df_rev,
    rev_sarima_noexog=rev_sarima_noexog,
    rev_sarima_exog=rev_sarima_exog,
    rev_ets_df=rev_ets_df,
    rev_prophet_df=rev_prophet_df,
    rev_theta_df=rev_theta_df,
    rev_lstm_df=rev_lstm_df
)

print(f"\n✓ 최종 매출 DataFrame 생성 완료")
print(f"  전체 행 수: {len(rev_final)}")
print(f"  기간: {rev_final.index.min().date()} ~ {rev_final.index.max().date()}")
print(f"  컬럼: {list(rev_final.columns)}")

print("\n📊 최근 데이터 (실제값 + 예측값 + TTM):")
display(rev_final.tail(10))

# 2026년 데이터만 확인
rev_2026 = rev_final[(rev_final.index >= '2026-01-01') & (rev_final.index < '2027-01-01')]
if not rev_2026.empty:
    print("\n📅 2026년 예측 데이터:")
    display(rev_2026)

### Step 7: 시가총액 데이터 수집

In [ ]:
print("=" * 70)
print("Step 7: 시가총액 데이터")
print("=" * 70)

print(f"\n💰 {ticker_with_a} 시가총액 데이터 수집 중...")
df_mc = get_market_cap_by_ticker(db_info, ticker_with_a)

if df_mc is None or df_mc.empty:
    print(f"⚠️  시가총액 데이터 없음!")
else:
    # 컬럼명 표준화
    if 'value' not in df_mc.columns:
        col_mapping = {col.lower(): col for col in df_mc.columns}
        possible_names = ['marketcap', 'market_cap', 'cap', 'mc', 'value']
        
        for name in possible_names:
            if name in col_mapping:
                df_mc = df_mc.rename(columns={col_mapping[name]: 'value'})
                break
    
    print(f"✓ 시가총액 데이터 수집 완료: {len(df_mc)}일")
    print(f"  기간: {df_mc.index.min().date()} ~ {df_mc.index.max().date()}")
    print(f"\n최근 시가총액:")
    display(df_mc.tail())

### Step 8: PSR 시계열 생성

In [ ]:
print("=" * 70)
print("Step 8: PSR 시계열 생성")
print("=" * 70)

# PSR = 시가총액 / 매출(TTM)
psr_df = build_psr_series(df_mc=df_mc, df_rev=df_rev, mc_divisor=1.0)

if psr_df is None or psr_df.empty:
    print("⚠️  PSR 계산 실패!")
else:
    print(f"✓ PSR 시계열 생성 완료: {len(psr_df)}일")
    print(f"  기간: {psr_df.index.min().date()} ~ {psr_df.index.max().date()}")
    print(f"\n최근 PSR:")
    display(psr_df.tail(10))
    
    # 통계
    print(f"\n📈 PSR 통계:")
    print(psr_df.describe())

### Step 9: 월별 외생변수 생성 (PSR 예측용)

In [ ]:
print("=" * 70)
print("Step 9: 월별 외생변수")
print("=" * 70)

exog_monthly_df = None

if HS_CODE:
    try:
        print(f"\n📦 월별 외생변수 생성 중...")
        
        # HS Code 데이터 수집 (db_info 파라미터 추가)
        df_export = get_hscode_processed_data(db_info, hs_code=HS_CODE)
        
        if df_export is not None and isinstance(df_export, dict):
            df_exog_monthly = extract_monthly_exog_var(df_export)
            
            if isinstance(df_exog_monthly, pd.DataFrame) and 'exog_var' in df_exog_monthly.columns:
                exog_monthly_df = _transform_exog_col(
                    df=df_exog_monthly[['exog_var']],
                    col='exog_var',
                    periods=12,
                    method="pct"
                )
                
                print(f"✓ 월별 외생변수 생성 완료: {len(exog_monthly_df)}개월")
                print(f"\n최근 데이터:")
                display(exog_monthly_df.tail())
            else:
                print("⚠️  월별 외생변수 생성 실패")
        else:
            print("⚠️  수출 데이터 없음")
    except Exception as e:
        print(f"⚠️  월별 외생변수 생성 실패: {e}")
        import traceback
        traceback.print_exc()
else:
    print("ℹ️  HS Code가 없어 외생변수를 사용하지 않습니다.")

### Step 10: PSR 예측

In [ ]:
print("=" * 70)
print("Step 10: PSR 예측")
print("=" * 70)

psr_horizon_months = FORECAST_CONFIG['psr_horizon_months']
psr_start_date = FORECAST_CONFIG.get('psr_start_date')

print(f"\n예측 월 수: {psr_horizon_months}")
print(f"예측 시작일: {psr_start_date if psr_start_date else '자동'}")

print("\n🔮 PSR 예측 실행 중...")
fc_table = forecast_psr_all_models(
    psr_df,
    horizon=psr_horizon_months,
    exog_df=exog_monthly_df,
    forecast_start_date=psr_start_date
)

if fc_table is None or fc_table.empty:
    print("⚠️  PSR 예측 실패!")
else:
    print(f"✓ PSR 예측 완료: {len(fc_table)}일")
    print(f"  기간: {fc_table.index.min().date()} ~ {fc_table.index.max().date()}")
    print(f"  컬럼: {list(fc_table.columns)}")
    print(f"\n📊 PSR 예측 결과:")
    display(fc_table.head(20))
    
    # 통계
    print(f"\n📈 PSR 예측 통계:")
    display(fc_table.describe())

### Step 11: Valuation 계산

In [ ]:
print("=" * 70)
print("Step 11: Valuation 계산")
print("=" * 70)

print("\n💵 Valuation 계산 중...")
valuation_forecast_result = compute_valuation_forecast(
    fc_table=fc_table,
    rev_final=rev_final,
    value_start_date=VALUE_START_DATE
)

if valuation_forecast_result is None or valuation_forecast_result.empty:
    print("⚠️  Valuation 계산 실패!")
else:
    print(f"✓ Valuation 계산 완료: {len(valuation_forecast_result)}일")
    print(f"  기간: {valuation_forecast_result.index.min().date()} ~ {valuation_forecast_result.index.max().date()}")
    print(f"  컬럼: {list(valuation_forecast_result.columns)}")
    
    print(f"\n📊 Valuation 결과 (최근 20일):")
    display(valuation_forecast_result.tail(20))
    
    # 통계
    print(f"\n📈 Valuation 통계:")
    display(valuation_forecast_result.describe())

### Step 12: 최종 결과 요약

In [ ]:
print("=" * 70)
print("최종 결과 요약")
print("=" * 70)

print(f"\n🎯 분석 종목: {ticker_with_a}")
print(f"\n📊 데이터 현황:")
print(f"  매출 데이터: {len(df_rev)}개 분기 ({df_rev.index.min().date()} ~ {df_rev.index.max().date()})")
print(f"  시가총액: {len(df_mc) if df_mc is not None else 0}일")
print(f"  PSR: {len(psr_df) if psr_df is not None else 0}일")

print(f"\n🔮 예측 결과:")
print(f"  매출 예측: {len(rev_final)}행")
print(f"  PSR 예측: {len(fc_table) if fc_table is not None else 0}일")
print(f"  Valuation: {len(valuation_forecast_result) if valuation_forecast_result is not None else 0}일")

print(f"\n✅ 분석 완료!")
print(f"\n💡 Tip: 위의 각 DataFrame을 변수명으로 직접 접근하여 추가 분석 가능")
print(f"   - df_rev: 원본 매출 데이터")
print(f"   - revenue_series: PeriodIndex 매출 시계열")
print(f"   - rev_final: 최종 매출 (실제 + 예측 + TTM)")
print(f"   - df_mc: 시가총액")
print(f"   - psr_df: PSR 시계열")
print(f"   - fc_table: PSR 예측")
print(f"   - valuation_forecast_result: 최종 Valuation")

---
## 9. 추가 분석 예시

위에서 생성된 데이터프레임들을 활용한 추가 분석 예시입니다.


### 9.1 모델별 예측값 비교

In [ ]:
# 2026년 매출 예측값 비교
if 'rev_final' in dir() and not rev_final.empty:
    rev_2026 = rev_final[(rev_final.index >= '2026-01-01') & (rev_final.index < '2027-01-01')]
    
    if not rev_2026.empty:
        # 예측 컬럼만 선택
        forecast_cols = [col for col in rev_2026.columns if 'revenue' in col and col != 'revenue_actual']
        
        if forecast_cols:
            print("2026년 모델별 매출 예측 비교:")
            display(rev_2026[forecast_cols])
            
            # 평균 예측값
            print("\n평균 예측값:")
            print(rev_2026[forecast_cols].mean(axis=1))

### 9.2 시각화 (선택 사항)

In [ ]:
# matplotlib 사용 시
# import matplotlib.pyplot as plt
# 
# # 매출 실제값 vs 예측값
# plt.figure(figsize=(15, 6))
# plt.plot(df_rev.index, df_rev['revenue'], label='Actual', marker='o')
# 
# if 'rev_ets_df' in dir() and rev_ets_df is not None:
#     plt.plot(rev_ets_df.index, rev_ets_df['revenue'], label='ETS', marker='s')
# 
# plt.legend()
# plt.title(f'{ticker_with_a} Revenue Forecast')
# plt.xlabel('Date')
# plt.ylabel('Revenue')
# plt.grid(True)
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()

print("시각화 코드는 주석을 해제하여 사용하세요.")

### 9.3 Excel 내보내기 (선택 사항)

In [ ]:
# Excel 파일로 내보내기
# output_file = f'{ticker_with_a}_analysis_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
# 
# with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
#     if 'df_rev' in dir():
#         df_rev.to_excel(writer, sheet_name='Revenue_Raw')
#     
#     if 'rev_final' in dir():
#         rev_final.to_excel(writer, sheet_name='Revenue_Final')
#     
#     if 'psr_df' in dir() and psr_df is not None:
#         psr_df.to_excel(writer, sheet_name='PSR')
#     
#     if 'fc_table' in dir() and fc_table is not None:
#         fc_table.to_excel(writer, sheet_name='PSR_Forecast')
#     
#     if 'valuation_forecast_result' in dir() and valuation_forecast_result is not None:
#         valuation_forecast_result.to_excel(writer, sheet_name='Valuation')
# 
# print(f"Excel 파일 저장 완료: {output_file}")

print("Excel 내보내기 코드는 주석을 해제하여 사용하세요.")